In [13]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

random_state = 42


In [15]:
data_dir = "/Users/shouryakulkarni/IDX-Exchange-Data-Science"
months = ["202510", "202511", "202512", "202601", "202602", "202603", "202604"]

frames = []
for m in months:
    f = pd.read_csv(f"{data_dir}/CRMLSSold{m}.csv", low_memory=False,
                    dtype={"PostalCode": str})
    f["DataMonth"] = f"{m[:4]}-{m[4:]}"
    frames.append(f)

raw = pd.concat(frames, ignore_index=True)
raw["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")

df = raw[(raw["PropertyType"] == "Residential") &
         (raw["PropertySubType"] == "SingleFamilyResidence")].copy()

In [3]:
data_dir = "/Users/shouryakulkarni/IDX-Exchange-Data-Science"
months = ["202510", "202511", "202512", "202601", "202602", "202603", "202604"]

frames = []
for m in months:
    f = pd.read_csv(f"{data_dir}/CRMLSSold{m}.csv", low_memory=False,
                    dtype={"PostalCode": str})
    f["DataMonth"] = f"{m[:4]}-{m[4:]}"   
    frames.append(f)

raw = pd.concat(frames, ignore_index=True)
raw["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")

df = raw[(raw["PropertyType"] == "Residential") &
         (raw["PropertySubType"] == "SingleFamilyResidence")].copy()

In [16]:
#documentation for how many rows each step removes 
cleaning_log = []
def log_step(name, before, after):
    removed = before - after
    pct = 100 * removed / before if before else 0
    cleaning_log.append({"step": name, "rows_before": before, "rows_after": after,
                         "rows_removed": removed, "pct_removed": round(pct, 2)})
    print(f"{name}: {before:,} -> {after:,}  (removed {removed:,}, {pct:.2f}%)")

log_step("Residential / SingleFamilyResidence filter", len(raw), len(df))

Residential / SingleFamilyResidence filter: 145,989 -> 72,724  (removed 73,265, 50.19%)


In [17]:
print(df["MlsStatus"].value_counts(dropna=False))

before = len(df)
df = df[df["MlsStatus"] == "Closed"]
log_step("MlsStatus == Closed", before, len(df))

MlsStatus
Closed    72724
Name: count, dtype: int64
MlsStatus == Closed: 72,724 -> 72,724  (removed 0, 0.00%)


In [18]:
before = len(df)
df = df[df["ClosePrice"].notna() & (df["ClosePrice"] > 0)]
log_step("ClosePrice not null and > 0", before, len(df))

ClosePrice not null and > 0: 72,724 -> 72,724  (removed 0, 0.00%)


In [19]:
for c in ["ListingContractDate", "PurchaseContractDate"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

bad_order = (
    (df["CloseDate"] < df["PurchaseContractDate"]) |
    (df["PurchaseContractDate"] < df["ListingContractDate"]) |
    (df["CloseDate"] < df["ListingContractDate"])
)
print("Rows with dates out of order:", bad_order.sum())

before = len(df)
df = df[~bad_order]
log_step("Dates in order (list <= contract <= close)", before, len(df))

Rows with dates out of order: 35
Dates in order (list <= contract <= close): 72,724 -> 72,689  (removed 35, 0.05%)


In [20]:
before = len(df)
df = df.sort_values("CloseDate").drop_duplicates(subset="ListingKey", keep="last")
log_step("Duplicate ListingKey", before, len(df))

df["AddressNorm"] = (df["UnparsedAddress"].astype(str).str.upper()
                     .str.replace(r"[^A-Z0-9 ]", "", regex=True)
                     .str.replace(r"\s+", " ", regex=True).str.strip())
before = len(df)
df = df.drop_duplicates(subset=["AddressNorm", "CloseDate"], keep="last")
log_step("Duplicate address + CloseDate", before, len(df))

resales = df[df.duplicated(subset="AddressNorm", keep=False)]
print("Addresses with more than one sale (kept):", resales["AddressNorm"].nunique())

Duplicate ListingKey: 72,689 -> 72,641  (removed 48, 0.07%)
Duplicate address + CloseDate: 72,641 -> 72,583  (removed 58, 0.08%)
Addresses with more than one sale (kept): 298


In [21]:
df["CloseMonth"] = df["CloseDate"].dt.to_period("M").astype(str)
print("Rows where file month != close month:", (df["CloseMonth"] != df["DataMonth"]).sum())

before = len(df)
df = df[df["CloseMonth"].isin(df["DataMonth"].unique())]
log_step("CloseMonth within downloaded months", before, len(df))
df["CloseMonth"].value_counts().sort_index()

Rows where file month != close month: 0
CloseMonth within downloaded months: 72,583 -> 72,583  (removed 0, 0.00%)


CloseMonth
2025-10    12008
2025-11     9710
2025-12    10435
2026-01     7476
2026-02     8942
2026-03    11546
2026-04    12466
Name: count, dtype: int64

Listing the columns that the model is allowed/not allowed to use.

| Feature | Verdict | Reason |
|---|---|---|
| ListPrice, OriginalListPrice | Exclude | Encodes the answer; doesn't exist for off-market homes |
| DaysOnMarket | Exclude | Only exists once a sale process has happened |
| PurchaseContractDate, ContractStatusChangeDate | Exclude | Recorded during/after the sale |
| TaxAnnualAmount | Exclude for now | Re-assessment can follow a sale; timing can't be verified |
| Agent / office fields | Exclude | Proxy for pricing strategy, not the house |
| ListingKey, ListingId, address | Exclude | Identifiers |
| MLSAreaMajor | Exclude | ~14% missing + ~6,000 "Not Defined"; ZIP/City cover it |
| Living area, beds, baths, lot, year built, garage, stories, pool/view/fireplace | Keep | Physical facts known before any sale |
| Lat/Long, County, City, ZIP | Keep | Fixed location |


In [23]:
target = "ClosePrice"

numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSqFt",
    "YearBuilt", "Latitude", "Longitude", "GarageSpaces", "Stories",
]
binary_features = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN"]
low_card_categorical = ["CountyOrParish"]
high_card_categorical = ["City", "PostalCode"]

for c in ["ListPrice", "OriginalListPrice", "TaxAnnualAmount", "LivingArea"]:
    if c in df.columns:
        print(f"corr(ClosePrice, {c}) = {df[target].corr(df[c]):.3f}")

corr(ClosePrice, ListPrice) = 0.188
corr(ClosePrice, OriginalListPrice) = 0.036
corr(ClosePrice, TaxAnnualAmount) = nan
corr(ClosePrice, LivingArea) = 0.116


In [24]:
before = len(df)
df = df[df["LivingArea"] > 0]
log_step("LivingArea > 0 and not missing", before, len(df))

for col in ["BedroomsTotal", "BathroomsTotalInteger"]:
    n = (df[col] == 0).sum()
    df.loc[df[col] == 0, col] = np.nan
    print(f"{col}: {n} zeros -> NaN")

min_sqft_per_bed = 100
before = len(df)
df = df[~(df["LivingArea"] / df["BedroomsTotal"] < min_sqft_per_bed)]
log_step(f"At least {min_sqft_per_bed} sq ft per bedroom", before, len(df))

bad_year = (df["YearBuilt"] < 1850) | (df["YearBuilt"] > df["CloseDate"].dt.year + 1)
print("Bad YearBuilt -> NaN:", bad_year.sum())
df.loc[bad_year, "YearBuilt"] = np.nan

df.loc[df["GarageSpaces"] > 20, "GarageSpaces"] = np.nan
df.loc[(df["Stories"] <= 0) | (df["Stories"] > 6), "Stories"] = np.nan

LivingArea > 0 and not missing: 72,583 -> 72,526  (removed 57, 0.08%)
BedroomsTotal: 31 zeros -> NaN
BathroomsTotalInteger: 20 zeros -> NaN
At least 100 sq ft per bedroom: 72,526 -> 72,524  (removed 2, 0.00%)
Bad YearBuilt -> NaN: 3


In [25]:
lc = df[["LotSizeAcres", "LotSizeSquareFeet"]].dropna()
lc = lc[lc["LotSizeSquareFeet"] > 0]
for rtol in [1e-5, 0.01, 0.02, 0.05]:
    share = np.isclose(lc["LotSizeSquareFeet"], lc["LotSizeAcres"] * 43560, rtol=rtol).mean()
    print(f"rtol={rtol:<6} -> {share:.1%} consistent")

rtol=1e-05  -> 14.8% consistent
rtol=0.01   -> 99.8% consistent
rtol=0.02   -> 99.8% consistent
rtol=0.05   -> 99.8% consistent


In [ ]:
df["LotSqFt"] = df["LotSizeSquareFeet"].where(df["LotSizeSquareFeet"] > 0)

unit_error = (
    (df["LotSizeAcres"] > 1_000) &
    np.isclose(df["LotSizeSquareFeet"], df["LotSizeAcres"] * 43560, rtol=0.02)
)
print("Lot unit errors fixed:", unit_error.sum())
df.loc[unit_error, "LotSqFt"] = df.loc[unit_error, "LotSizeAcres"]

fill_from_acres = df["LotSqFt"].isna() & df["LotSizeAcres"].between(0.01, 1_000)
df.loc[fill_from_acres, "LotSqFt"] = df.loc[fill_from_acres, "LotSizeAcres"] * 43560
print("Missing lot sizes filled from acres:", fill_from_acres.sum())

df.loc[df["LotSqFt"] > 1_000 * 43560, "LotSqFt"] = np.nan
df["LotSqFt"].describe()